In [6]:
from pathlib import Path
import yaml
import numpy as np
import pandas as pd

FIELDS = ["Trigger event", "Direct Consequences", "Aftermaths"]

def parse_tokens(value):
    # Your example: "a, b, c" in a single YAML scalar
    if value is None:
        return []
    if isinstance(value, list):
        out = []
        for v in value:
            out.extend(parse_tokens(v))
        return out
    if isinstance(value, str):
        return [t.strip() for t in value.split(",") if t.strip()]
    s = str(value).strip()
    return [s] if s else []

# ---- Load works as sets of atomic attributes ----
folder = Path("C:/Users/Anwender/Documents/Epicode_Python/co-occurence_prepararsi_al_presente/works/works_chatgpt")  # <-- change this
works = []

for fp in sorted(folder.glob("*.yml")):
    with fp.open("r", encoding="utf-8") as f:
        data = yaml.safe_load(f) or {}

    attrs = set()
    for field in FIELDS:
        attrs.update(parse_tokens(data.get(field)))

    works.append(sorted(attrs))

# ---- Vocabulary ----
vocab = sorted({a for attrs in works for a in attrs})
idx = {a: i for i, a in enumerate(vocab)}

# ---- Build incidence matrix X (works x attributes) ----
X = np.zeros((len(works), len(vocab)), dtype=np.int32)
for r, attrs in enumerate(works):
    for a in attrs:
        X[r, idx[a]] = 1

# ---- Co-occurrence: attributes x attributes ----
C = X.T @ X   # C[i,j] = number of works containing both i and j

# ---- Export co-occurrence matrix with labels ----
co_df = pd.DataFrame(C, index=vocab, columns=vocab)
co_df.to_csv("chatgpt_cooccurrence_atomic_attributes_jp.csv", encoding="utf-8")
print("Wrote chatgpt_cooccurrence_atomic_attributes_jp.csv with shape:", co_df.shape)

Wrote chatgpt_cooccurrence_atomic_attributes_jp.csv with shape: (48, 48)
